In [17]:
import kagglehub
data_path = kagglehub.dataset_download("azrilbagas/pdf-file-rag")

In [18]:
!pip install PyPDF2
!pip install faiss-cpu -q

In [19]:
import os
pdf_files = [
    os.path.join(root, f)
    for root, _, files in os.walk(data_path)
    for f in files
    if f.lower().endswith(".pdf")
]
print(pdf_files)

['/kaggle/input/datasets/azrilbagas/pdf-file-rag/iso27001.pdf']


In [20]:
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [21]:
def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"
    return full_text

In [22]:
def chunk_text(text , chunk_size=500 , overlap=50):
    words = text.split()
    chunks = []
    for i in range(0 , len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [23]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [24]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

In [25]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [26]:
def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [27]:
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    print(inputs["input_ids"].device)
    print(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [28]:
pdf_path = pdf_files[0]
text = extract_text(pdf_path)
text = clean_text(text)
chunks = chunk_text(text, chunk_size=120, overlap=30)
model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
Q = input("Enter your question :\n")
question = Q
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)
context = "\n\n".join(top_chunks)
for i, chunk in enumerate(top_chunks, 1):
    print(f"\n Answer {i} \n{chunk}")

Enter your question :
 What does ISO/IEC 27001:2022 specify requirements for?



 Answer 1 
2022 – All rights reserved Contents Page --``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`--- ISO/IEC 27001:2022(E) Foreword ISO (the International Organization for Standardization) and IEC (the International Electrotechnical Commission) form the specialized system for worldwide standardization. National bodies that are members of ISO or IEC participate in the development of International Standards through technical committees established by the respective organization to deal with particular fields of technical activity. ISO and IEC technical committees collaborate in fields of mutual interest. Other international organizations, governmental and non-governmental, in liaison with ISO and IEC, also take part in the work. The procedures used to develop this document and those intended for its further maintenance are described in the ISO/IEC Directives, Part 1. In particular, the different approval criteria needed

 Answer 2 
some or all of their content constitutes requirements 

In [39]:
chunk = top_chunks[:3]
prompt =  f"""Answer the question in one short natural sentence based only on the text and show the final answer only \n
Text:
{context}\n
Question: {question}\n


Answer:"""

answer = generate_text(prompt, max_length=650)
print(answer[0])

cuda:0
cuda:0
Answer the question in one short natural sentence based only on the text and show the final answer only 

Text:
2022 – All rights reserved Contents Page --``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`--- ISO/IEC 27001:2022(E) Foreword ISO (the International Organization for Standardization) and IEC (the International Electrotechnical Commission) form the specialized system for worldwide standardization. National bodies that are members of ISO or IEC participate in the development of International Standards through technical committees established by the respective organization to deal with particular fields of technical activity. ISO and IEC technical committees collaborate in fields of mutual interest. Other international organizations, governmental and non-governmental, in liaison with ISO and IEC, also take part in the work. The procedures used to develop this document and those intended for its further maintenance are described in the ISO/IEC Directives, Part 1. In par